# Step 3: Hybrid CNN + ViT Model Training
## China Fundus CIMT Dataset — CIMT Thickening Classification

### Architecture
- **CNN Branch**: ResNet50 (pretrained, ImageNet) → GAP → 2048 features
- **ViT Branch**: ViT-B/16 (pretrained, ImageNet) → 768 features
- **Fusion**: Concat(2816) → Dense(512) → Dense(256) → Sigmoid

### Prerequisites
```
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
pip install timm
```

In [1]:
# BLOCK 1: IMPORTS
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
import timm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
PROCESSED_DIR = '../data/Retinal/processed/'
OUTPUT_DIR    = 'outputs/'
PLOTS_DIR     = 'outputs/plots/'
MODEL_DIR     = '../models/retinal/'
for d in [OUTPUT_DIR, PLOTS_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")

Device: cpu
Directories ready.


In [ ]:
# BLOCK 2: LOAD PREPROCESSED DATA
train_images  = np.load(os.path.join(PROCESSED_DIR, 'train_images.npy'))
train_labels  = np.load(os.path.join(PROCESSED_DIR, 'train_labels.npy'))
val_images    = np.load(os.path.join(PROCESSED_DIR, 'val_images.npy'))
val_labels    = np.load(os.path.join(PROCESSED_DIR, 'val_labels.npy'))
test_images   = np.load(os.path.join(PROCESSED_DIR, 'test_images.npy'))
test_labels   = np.load(os.path.join(PROCESSED_DIR, 'test_labels.npy'))
class_weights = np.load(os.path.join(PROCESSED_DIR, 'class_weights.npy'), allow_pickle=True)
print(f"Train: {train_images.shape}  Val: {val_images.shape}  Test: {test_images.shape}")
print(f"Class weights: Normal={class_weights[0]:.4f}  Thickened={class_weights[1]:.4f}")

In [ ]:
# BLOCK 3: DATASET AND DATALOADERS
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

class FundusDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = (images * 255).astype('uint8')
        self.labels    = labels.astype('int64')
        self.transform = transform
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = self.transform(self.images[idx]) if self.transform else self.images[idx]
        return img, self.labels[idx]

BATCH_SIZE = 32
sample_weights = np.array([class_weights[l] for l in train_labels])
sampler = WeightedRandomSampler(torch.from_numpy(sample_weights).float(), len(sample_weights), replacement=True)
train_loader = DataLoader(FundusDataset(train_images, train_labels, train_transform), batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader   = DataLoader(FundusDataset(val_images, val_labels, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(FundusDataset(test_images, test_labels, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train batches: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}")

In [ ]:
# BLOCK 4: HYBRID CNN + VIT MODEL
class HybridCNNViT(nn.Module):
    def __init__(self, dropout1=0.3, dropout2=0.2):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.cnn_branch = nn.Sequential(*list(resnet.children())[:-1])  # (B,2048,1,1)
        self.vit_branch = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(
            nn.Linear(2048+768, 512), nn.ReLU(), nn.Dropout(dropout1),
            nn.Linear(512, 256),     nn.ReLU(), nn.Dropout(dropout2),
            nn.Linear(256, 1),       nn.Sigmoid()
        )
    def forward(self, x):
        cnn_feat = self.cnn_branch(x).view(x.size(0), -1)
        vit_feat = self.vit_branch(x)
        return self.classifier(torch.cat([cnn_feat, vit_feat], dim=1)).squeeze(1)

model = HybridCNNViT().to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Hybrid CNN+ViT | Params: {total:,} | CNN:2048 | ViT:768 | Fusion:2816")

In [ ]:
# BLOCK 5: TRAINING CONFIG
LEARNING_RATE = 1e-4
EPOCHS        = 50
PATIENCE      = 10
MODEL_PATH    = os.path.join(MODEL_DIR, 'hybrid_model.pth')
cw_tensor     = torch.tensor(class_weights, dtype=torch.float32).to(device)

def weighted_bce_loss(outputs, targets):
    weights = cw_tensor[targets.long()]
    return (nn.BCELoss(reduction='none')(outputs, targets.float()) * weights).mean()

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)
print(f"Optimizer: Adam lr={LEARNING_RATE} | EarlyStopping patience={PATIENCE} | Model: {MODEL_PATH}")

In [ ]:
# BLOCK 6: TRAINING LOOP
def train_epoch(model, loader, optimizer):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = weighted_bce_loss(out, labels.float())
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += ((out >= 0.5).long() == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum/total, correct/total

def eval_epoch(model, loader):
    model.eval()
    loss_sum, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = weighted_bce_loss(out, labels.float())
            loss_sum += loss.item() * imgs.size(0)
            correct  += ((out >= 0.5).long() == labels).sum().item()
            total    += imgs.size(0)
    return loss_sum/total, correct/total

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_loss, patience_count = float('inf'), 0
start_time = time.time()

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}")
print("-"*55)
for epoch in range(1, EPOCHS+1):
    tl, ta = train_epoch(model, train_loader, optimizer)
    vl, va = eval_epoch(model, val_loader)
    scheduler.step(vl)
    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)
    print(f"{epoch:>5} | {tl:>10.4f} | {ta:>9.4f} | {vl:>8.4f} | {va:>7.4f}")
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"        *** Saved best model (val_loss={best_val_loss:.4f}) ***")
        patience_count = 0
    else:
        patience_count += 1
    if patience_count >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

training_time = time.time() - start_time
best_val_acc  = max(history['val_acc'])
print(f"\nDone in {training_time/60:.1f} min | Best val acc: {best_val_acc*100:.2f}%")

In [ ]:
# BLOCK 7: TRAINING CURVES
epochs_range = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train', markersize=3)
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val',   markersize=3)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train', markersize=3)
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Val',   markersize=3)
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150)
plt.show()
print("Training curves saved.")

In [ ]:
# BLOCK 8: EVALUATE ON TEST SET
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
all_preds, all_probs, all_labels_list = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        out = model(imgs.to(device)).cpu().numpy()
        all_preds.extend((out >= 0.5).astype(int))
        all_probs.extend(out)
        all_labels_list.extend(labels.numpy())
all_preds = np.array(all_preds); all_probs = np.array(all_probs); all_labels_list = np.array(all_labels_list)

test_acc  = accuracy_score(all_labels_list, all_preds)
test_prec = precision_score(all_labels_list, all_preds, zero_division=0)
test_rec  = recall_score(all_labels_list, all_preds, zero_division=0)
test_f1   = f1_score(all_labels_list, all_preds, zero_division=0)
test_auc  = roc_auc_score(all_labels_list, all_probs)
print(f"Test Accuracy: {test_acc*100:.2f}%  Precision: {test_prec*100:.2f}%  Recall: {test_rec*100:.2f}%  F1: {test_f1*100:.2f}%  AUC: {test_auc*100:.2f}%")
print(classification_report(all_labels_list, all_preds, target_names=['Normal','Thickened']))

In [ ]:
# BLOCK 9: CONFUSION MATRIX + ROC
cm = confusion_matrix(all_labels_list, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal','Thickened'], yticklabels=['Normal','Thickened'])
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
fpr, tpr, _ = roc_curve(all_labels_list, all_probs)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={test_auc:.4f}')
axes[1].plot([0,1],[0,1],'k--'); axes[1].set_title('ROC Curve')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.savefig(os.path.join(PLOTS_DIR, 'roc_curve.png'), dpi=150)
plt.show()
tn,fp,fn,tp = cm.ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

In [ ]:
# BLOCK 10: SAVE RESULTS
results_df = pd.DataFrame([{
    'Model':'Hybrid ResNet50+ViT-B/16', 'Test_Accuracy':round(test_acc,4),
    'Test_Precision':round(test_prec,4), 'Test_Recall':round(test_rec,4),
    'Test_F1':round(test_f1,4), 'Test_AUC':round(test_auc,4),
    'Best_Val_Accuracy':round(best_val_acc,4), 'Training_Time_min':round(training_time/60,2),
    'Epochs_Trained':len(history['train_loss'])
}])
results_df.to_csv(os.path.join(OUTPUT_DIR, 'model_results.csv'), index=False)
pd.DataFrame(classification_report(all_labels_list, all_preds,
    target_names=['Normal','Thickened'], output_dict=True)).transpose().to_csv(
    os.path.join(OUTPUT_DIR, 'classification_report.csv'))
print("="*55)
print(f"Test Accuracy:    {test_acc*100:.2f}%")
print(f"Test AUC:         {test_auc*100:.2f}%")
print(f"Best Val Acc:     {best_val_acc*100:.2f}%")
print(f"Training Time:    {training_time/60:.1f} min")
print(f"Model saved:      {MODEL_PATH}")
print("="*55)